# TDA replication - analysis

Plots and analyses of the results produced by `experiments.py` (cluster or Colab). **No GPU needed**: it only
reads files, so it runs on your own computer as well (it needs `pandas` and `matplotlib`).

Inputs, in the `results/` folder:
- `runs.jsonl`: one line per (run, dataset) with the final accuracy, CLIP's accuracy on the same stream and the settings;
- `traces/*.csv.gz`: one file per (run, dataset) describing **every test image**:

| column | meaning |
|---|---|
| `step` | position in the stream (0 = first image) |
| `label`, `clip_pred`, `pred` | true class, zero-shot CLIP prediction, TDA prediction |
| `clip_correct`, `correct` | 1 if the prediction is right |
| `entropy` | CLIP entropy normalised by log2 N (the value gated by the negative cache) |
| `pos_update`, `neg_update` | what happened to the caches: `add`, `replace` or empty (nothing) |
| `pos_size`, `neg_size` | number of entries in each cache after this image |
| `pos_classes` | number of classes with at least one entry in the positive cache |

Every section works on whatever results exist and says so when something is missing. Figures are saved in
`results/figures/`.

## 0. Load the results
Set `RESULTS` to the folder copied from the cluster (`scp -r ...:~/TDA_replication/results .`) or to the Drive
folder on Colab.

In [ ]:
import glob, json, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RESULTS = "results"            # e.g. r"C:\Users\me\Downloads\tda_results" or "/content/drive/MyDrive/TDA_replication/results"
FIG = os.path.join(RESULTS, "figures")
os.makedirs(FIG, exist_ok=True)
plt.rcParams.update({"figure.figsize": (7, 3.5), "axes.grid": True, "grid.alpha": 0.3})

rows = [json.loads(l) for l in open(os.path.join(RESULTS, "runs.jsonl")) if l.strip()]
runs = pd.DataFrame(rows)
for col, default in [("impl", "replica"), ("mode", "tda"), ("seed", 1), ("order", "official"), ("limit", None),
                     ("clip_acc", np.nan), ("trace", None)]:
    if col not in runs:
        runs[col] = default
runs["impl"] = runs["impl"].fillna("replica")
runs["overrides"] = runs["overrides"].apply(lambda o: o if isinstance(o, dict) else {})
runs = runs[runs["limit"].isna()].copy()                       # drop smoke tests / equivalence test
runs["setting"] = runs.apply(lambda r: " ".join(
    [f"mode={r['mode']}"] * (r["mode"] != "tda") + [f"order={r['order']}"] * (r["order"] != "official")
    + [f"{k}={v}" for k, v in sorted(r["overrides"].items())]) or "default", axis=1)
runs["gain"] = runs["acc"] - runs["clip_acc"]
# keep the latest result of identical runs
key = ["experiment", "impl", "dataset", "backbone", "setting", "seed"]
runs = runs.drop_duplicates(key, keep="last").reset_index(drop=True)


def load_trace(run):
    # Per-image trace of one row of `runs` (a Series), or None if it was not saved.
    if not isinstance(run.get("trace"), str):
        return None
    t = pd.read_csv(os.path.join(RESULTS, "traces", run["trace"]))
    t[["pos_update", "neg_update"]] = t[["pos_update", "neg_update"]].fillna("")
    return t


def pick(**filters):
    # Rows of `runs` matching every filter, e.g. pick(experiment="paper_cd", dataset="dtd", backbone="RN50").
    sel = runs
    for k, v in filters.items():
        sel = sel[sel[k] == v]
    return sel


print(f"{len(runs)} results, experiments: {sorted(runs.experiment.unique())}")
runs[["experiment", "impl", "backbone", "dataset", "setting", "seed", "acc", "clip_acc", "gain"]].tail(10)

## 1. Overview
Accuracy of every setting of every experiment (mean over seeds) and the gain over zero-shot CLIP on the same
stream. Rows are datasets, columns are settings.

In [ ]:
rep = runs[runs.impl == "replica"]
for (exp, bb), g in rep.groupby(["experiment", "backbone"]):
    table = g.pivot_table(index="dataset", columns="setting", values="acc", aggfunc="mean")
    table.insert(0, "CLIP", g.groupby("dataset")["clip_acc"].mean())
    print(f"\n=== {exp} | {bb} ===")
    display(table.round(2))

## 2. Accuracy along the stream
TDA starts with empty caches and improves while it sees more images. The left plot shows the **running
accuracy** (accuracy over the first *t* images) of CLIP and TDA; the right plot the **gain in a moving window**
of `WINDOW` images, which shows *when* in the stream the caches start to help.

Choose the run with `RUN` (any row of `runs`; by default the first replica run of `paper_cd` with RN50).

In [ ]:
WINDOW = 200
cand = pick(experiment="paper_cd", impl="replica", backbone="RN50", setting="default")
cand = cand[cand.trace.notna()]
RUN = (cand if len(cand) else runs[runs.trace.notna()]).iloc[0]       # or e.g. pick(dataset="eurosat", experiment="seeds").iloc[0]
print("RUN:", RUN.experiment, RUN.dataset, RUN.backbone, RUN.setting, "seed", RUN.seed)

def plot_stream(run, window=WINDOW, ax=None):
    t = load_trace(run)
    if t is None:
        print("no trace for this run"); return
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.5))
    n = np.arange(1, len(t) + 1)
    a1.plot(100 * t.clip_correct.cumsum() / n, label="CLIP")
    a1.plot(100 * t.correct.cumsum() / n, label="TDA")
    a1.set(xlabel="images seen", ylabel="running accuracy (%)", title=f"{run.dataset} | {run.backbone} | {run.setting}")
    a1.legend()
    gain = 100 * (t.correct - t.clip_correct).rolling(window, min_periods=window // 4).mean()
    a2.plot(gain, color="tab:green"); a2.axhline(0, color="k", lw=0.8)
    a2.set(xlabel="images seen", ylabel=f"gain over CLIP, window {window} (pts)", title="When does TDA help?")
    fig.tight_layout()
    fig.savefig(os.path.join(FIG, f"stream_{run.experiment}_{run.dataset}_{run.backbone.replace('/', '-')}.png"), dpi=150)
    plt.show()

plot_stream(RUN)

## 3. Cache dynamics
How the two caches fill up and how often they change. With shot capacity *k* the positive cache holds at most
*k* entries per class, so it saturates once every class has been predicted *k* times; afterwards it only
**replaces** entries with more confident ones. The negative cache accepts only images with entropy in
(0.2, 0.5), so it fills more slowly.

In [ ]:
def plot_caches(run, window=WINDOW):
    t = load_trace(run)
    if t is None:
        print("no trace for this run"); return
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.5))
    a1.plot(t.pos_size, label="positive cache entries")
    a1.plot(t.neg_size, label="negative cache entries")
    a1.plot(t.pos_classes, "--", label="classes in positive cache")
    a1.set(xlabel="images seen", ylabel="count", title="Cache size")
    a1.legend()
    for col, name in [("pos_update", "positive"), ("neg_update", "negative")]:
        for act, ls in [("add", "-"), ("replace", ":")]:
            a2.plot(100 * (t[col] == act).rolling(window, min_periods=1).mean(), ls, label=f"{name} {act}")
    a2.set(xlabel="images seen", ylabel=f"% of images, window {window}", title="Cache updates")
    a2.legend(fontsize=8)
    fig.tight_layout()
    fig.savefig(os.path.join(FIG, f"caches_{run.experiment}_{run.dataset}_{run.backbone.replace('/', '-')}.png"), dpi=150)
    plt.show()

plot_caches(RUN)

## 4. When does TDA change the prediction?
Every image falls in one of four groups: both right, both wrong, **fixed** (CLIP wrong → TDA right) and
**broken** (CLIP right → TDA wrong). The net gain is *fixed − broken*. Splitting by CLIP's entropy shows
whether TDA helps mostly on confident or on uncertain images.

In [ ]:
def fixes_and_breaks(run, bins=(0, 0.1, 0.2, 0.3, 0.5, 1.0)):
    t = load_trace(run)
    if t is None:
        print("no trace for this run"); return
    t["group"] = np.select([(t.clip_correct == 0) & (t.correct == 1), (t.clip_correct == 1) & (t.correct == 0),
                            t.correct == 1], ["fixed", "broken", "both right"], "both wrong")
    t["entropy bin"] = pd.cut(t.entropy, bins=list(bins), include_lowest=True)
    table = pd.crosstab(t["entropy bin"], t["group"])
    table["net gain (pts)"] = 100 * (table.get("fixed", 0) - table.get("broken", 0)) / len(t)
    display(table)
    ax = table[[c for c in ["fixed", "broken"] if c in table]].plot.bar(rot=0, title=f"{run.dataset} | {run.setting}")
    ax.set(xlabel="CLIP entropy (normalised)", ylabel="images")
    plt.tight_layout(); plt.savefig(os.path.join(FIG, f"fixes_{run.experiment}_{run.dataset}.png"), dpi=150); plt.show()

fixes_and_breaks(RUN)

## 5. Stream order and seeds
TDA is online, so its accuracy depends on the order of the stream. With several seeds (experiment `seeds`) the
spread shows how much of a difference is just noise; comparing `order=random` and `order=sorted` (experiment
`stream_order`) shows what happens when the stream is not i.i.d. (images grouped by class).

In [ ]:
multi = runs[(runs.impl == "replica") & runs.groupby(["experiment", "dataset", "backbone", "setting"]).seed.transform("nunique").gt(1)]
if multi.empty:
    print("No experiment with several seeds yet (run e.g. `seeds` or `stream_order`).")
else:
    summary = multi.groupby(["experiment", "backbone", "dataset", "setting"]).agg(
        acc_mean=("acc", "mean"), acc_std=("acc", "std"), gain_mean=("gain", "mean"), seeds=("seed", "nunique"))
    display(summary.round(2))
    for (exp, bb), g in multi.groupby(["experiment", "backbone"]):
        fig, ax = plt.subplots(figsize=(8, 3.5))
        labels, data = [], []
        for (d, s), gg in g.groupby(["dataset", "setting"]):
            labels.append(f"{d}\n{s}"); data.append(gg.gain.values)
        ax.boxplot(data)
        ax.set_xticks(range(1, len(labels) + 1))
        ax.set_xticklabels(labels)
        ax.axhline(0, color="k", lw=0.8)
        ax.set(ylabel="gain over CLIP (pts)", title=f"{exp} | {bb}: spread over seeds")
        plt.xticks(fontsize=7); plt.tight_layout()
        plt.savefig(os.path.join(FIG, f"seeds_{exp}_{bb.replace('/', '-')}.png"), dpi=150); plt.show()

For a sorted stream, the running-accuracy plot of section 2 is especially telling: compare one run of each order.

In [ ]:
for order in ["random", "sorted"]:
    sel = runs[(runs.order == order) & runs.trace.notna()]
    if len(sel):
        plot_stream(sel.iloc[0])
    else:
        print(f"no run with order={order} yet")

## 6. Which classes benefit?
Accuracy of CLIP and TDA per class over the whole stream: the classes TDA fixes most and the ones it hurts.

In [ ]:
def per_class(run, top=10):
    t = load_trace(run)
    if t is None:
        print("no trace for this run"); return
    g = t.groupby("label").agg(images=("step", "size"), clip=("clip_correct", "mean"), tda=("correct", "mean"))
    g[["clip", "tda"]] *= 100
    g["gain"] = g["tda"] - g["clip"]
    print(f"{run.dataset}: {(g.gain > 0).sum()} classes improve, {(g.gain < 0).sum()} get worse, {(g.gain == 0).sum()} unchanged")
    display(pd.concat([g.nlargest(top, "gain"), g.nsmallest(top, "gain")]).round(1))

per_class(RUN)

## 7. Hyper-parameters
For experiments with a grid over two config values (e.g. `alpha_beta`), a heatmap of the mean accuracy over the
datasets of the experiment.

In [ ]:
def heatmap(experiment, x, y, backbone="RN50"):
    g = pick(experiment=experiment, backbone=backbone, impl="replica")
    if g.empty:
        print(f"experiment {experiment!r} not run yet"); return
    g = g.assign(**{x: g.overrides.apply(lambda o: o.get(x)), y: g.overrides.apply(lambda o: o.get(y))})
    table = g.pivot_table(index=y, columns=x, values="acc", aggfunc="mean")
    fig, ax = plt.subplots(figsize=(5, 3.5))
    im = ax.imshow(table.values, cmap="viridis", origin="lower")
    ax.set_xticks(range(len(table.columns))); ax.set_xticklabels(table.columns)
    ax.set_yticks(range(len(table.index))); ax.set_yticklabels(table.index)
    for i in range(table.shape[0]):
        for j in range(table.shape[1]):
            ax.text(j, i, f"{table.values[i, j]:.1f}", ha="center", va="center", color="w", fontsize=8)
    ax.set(xlabel=x, ylabel=y, title=f"{experiment}: mean accuracy (%)"); ax.grid(False)
    fig.colorbar(im); fig.tight_layout()
    fig.savefig(os.path.join(FIG, f"heatmap_{experiment}.png"), dpi=150); plt.show()

heatmap("alpha_beta", "positive.alpha", "positive.beta")